# Лабораторная работа №5
## Аппарат сетей Петри: параметризованная анимация

В данном скрипте строятся анимации изменения маркировки сети Петри
для задачи обедающих философов.

В отличие от базовой версии, здесь рассматриваются три набора параметров.
Для каждого набора создаётся отдельный GIF-файл.

Это позволяет наглядно сравнить поведение системы
при разном числе философов и разной длительности моделирования.

In [ ]:
using DrWatson
@quickactivate "project"

## Подключение модели и библиотек

Подключается модуль `DiningPhilosophers` из каталога `src`,
а также библиотека `Plots` для построения анимации.

In [ ]:
include(srcdir("DiningPhilosophers.jl"))
using .DiningPhilosophers

using Plots
using DataFrames

## Наборы параметров

Для анимации выбираются три набора параметров.
Это позволяет не перегружать проект большим числом GIF-файлов
и при этом провести небольшое сравнение.

In [ ]:
param_sets = [
    (N = 3, tmax = 20.0, fps = 5),
    (N = 4, tmax = 25.0, fps = 5),
    (N = 5, tmax = 30.0, fps = 5),
]

println("Запуск параметризованной генерации GIF-анимаций...")

## Построение анимаций

Для каждого набора параметров:
- создаётся классическая сеть Петри;
- выполняется стохастическая симуляция;
- по полученным данным строится анимация;
- результат сохраняется в папку `plots/`.

In [ ]:
for p in param_sets
    N = p.N
    tmax = p.tmax
    fps = p.fps

    println("Создание анимации: N = $N, tmax = $tmax, fps = $fps")

    net, u0, place_names = build_classical_network(N)
    df = simulate_stochastic(net, u0, tmax)

    anim = @animate for k in 1:nrow(df)
        values = [df[k, String(name)] for name in place_names]

        bar(
            string.(place_names),
            values;
            xlabel = "Позиции сети Петри",
            ylabel = "Число фишек",
            title = "Маркировка сети Петри, N = $N, t = $(round(df[k, :time], digits=2))",
            legend = false,
            xticks = :all,
            xrotation = 45,
            size = (1000, 500)
        )
    end

    filename = "philosophers_simulation_N$(N)_t$(Int(round(tmax))).gif"
    gif(anim, plotsdir(filename), fps = fps)

    println("Сохранено: ", plotsdir(filename))
end

## Результат

После выполнения скрипта в папке `plots/` будут созданы три GIF-файла:
- для `N = 4`, `tmax = 20`;
- для `N = 5`, `tmax = 25`;
- для `N = 6`, `tmax = 30`.

Каждая анимация показывает изменение маркировки сети Петри во времени.
Это позволяет визуально сравнить динамику системы
для разных параметров моделирования.

In [ ]:
println("Готово. Созданы 3 GIF-файла в папке plots/")